In [67]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime, time
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
import zipfile
import io
import statsmodels.api as sm
import os
import glob
from sqlalchemy import create_engine, types as satypes
import contextlib
import xml.etree.ElementTree as ET
from tqdm import tqdm


In [23]:
folder_path = os.path.join(os.path.expanduser("~"), "Desktop")
os.path.samefile(folder_path, os.getcwd()), time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

(True, '2025-10-05 01:28:24')

In [68]:
username = "postgres"
password = "GeorgeHighbury0725!!"
host = "localhost"
port = "5432"
database = "sp_500_OHLCV_database"

In [69]:
engine = create_engine(f"postgresql://{username}:{password}@{host}:{port}/{database}")
connection = engine.connect()

In [70]:
sp500_url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

try:
    wikipedia_tables = pd.read_html(sp500_url)
except Exception:
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(sp500_url, headers=headers)
    response.raise_for_status()
    wikipedia_tables = pd.read_html(io.StringIO(response.text))

wikipedia_table = wikipedia_tables[0]

wikipedia_table = wikipedia_tables[0].rename(columns={
    "Symbol": "ticker",
    "Security": "company_name",
    "GICS Sector": "sector",
    "GICS Sub-Industry": "sub_industry",
    "Headquarters Location": "headquarters_location",
    "Date added": "date_added",
    "CIK": "cik",
    "Founded": "founded"
})

wikipedia_table.to_sql("sp500_reference", engine, if_exists="replace", index=False)

503

In [71]:
sp500_reference = pd.read_sql("SELECT * FROM sp500_reference;", engine)
sp500_reference

,ticker,company_name,sector,sub_industry,headquarters_location,date_added,cik,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


In [72]:
symbol_column = sp500_reference["ticker"]

ticker_list = []
for ticker in symbol_column:
    ticker_list.append(ticker)

ticker_list = [ticker.replace(".", "-") for ticker in ticker_list]  # Wikipedia uses "." in some tickers to denote class A/B shares. Also this is an example of list comprehension.
ticker_list.sort()

In [98]:
with engine.connect() as conn:
    ohlcv_dates = pd.read_sql("""
        SELECT 
            MIN(date) AS earliest_date,
            MAX(date) AS latest_date
        FROM sp500_ohlcv;
    """, conn).iloc[0]

    edgar_dates = pd.read_sql("""
        SELECT 
            MIN(filed) AS earliest_date,
            MAX(filed) AS latest_date
        FROM sp500_edgar_financials;
    """, conn).iloc[0]

ohlcv_start = pd.to_datetime(ohlcv_dates["earliest_date"])
ohlcv_end   = pd.to_datetime(ohlcv_dates["latest_date"])
edgar_start = pd.to_datetime(edgar_dates["earliest_date"])
edgar_end   = pd.to_datetime(edgar_dates["latest_date"])

ohlcv_start, ohlcv_end, edgar_start, edgar_end

(Timestamp('2025-01-02 00:00:00'),
 Timestamp('2025-10-06 00:00:00'),
 Timestamp('2009-04-15 00:00:00'),
 Timestamp('2025-10-07 00:00:00'))

In [105]:
start_input = input("Start date (YYYY-MM-DD or 'start'): ").lower()
if start_input == "start":
    start = ohlcv_start
else:
    start = pd.Timestamp(datetime.datetime.strptime(start_input, "%Y-%m-%d")).normalize()
end_input = input("End date (YYYY-MM-DD or 'now'): ").lower()
if end_input == "now":
    end = pd.Timestamp.now().normalize()
else:
    end = pd.Timestamp(datetime.datetime.strptime(end_input, "%Y-%m-%d")).normalize()
    
start, end

(Timestamp('2025-01-02 00:00:00'), Timestamp('2025-10-07 00:00:00'))

In [9]:
sp500df = pd.read_sql("SELECT * FROM sp500_ohlcv;", engine, parse_dates=["date"]).pivot(index="date", columns="ticker", values=["open", "high", "low", "close", "adj_close", "volume"]).sort_index(axis=1, level=0)
sp500df.tail()

adj_close                                                  \
ticker               A        AAPL        ABBV        ABNB         ABT   
date                                                                     
2025-09-29  123.750000  254.429993  223.160004  122.919998  133.110001   
2025-09-30  128.350006  254.630005  231.539993  121.419998  133.940002   
2025-10-01  138.580002  255.449997  244.380005  122.320000  133.470001   
2025-10-02  138.699997  257.130005  236.559998  121.489998  132.990005   
2025-10-03  141.639999  258.019989  233.910004  120.220001  134.589996   

                                                                      ...  \
ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
date                                                                  ...   
2025-09-29  89.830002  247.000000  359.420013  244.789993  60.310001  ...   
2025-09-30  90.730003  246.600006  352.750000  245.699997  59.740002  ...   
2025-10-01  90.309998  243.710007  343.720001  239.279999  59.250000  ...   
2025-10-02  89.080002  244.339996  351.480011  241.669998  59.110001  ...   
2025-10-03  90.790001  245.320007  346.739990  241.990005  61.040001  ...   

               volume                                                          \
ticker             WY       WYNN        XEL         XOM        XYL        XYZ   
date                                                                            
2025-09-29  5111100.0  2014800.0  5527100.0  19189300.0  1087600.0  7237200.0   
2025-09-30  5465100.0  1511300.0  4011000.0  18076200.0  1841000.0  7838100.0   
2025-10-01  3650300.0  1532600.0  4585600.0  16614000.0  1321000.0  6640700.0   
2025-10-02  3592700.0  1286100.0  6982500.0  13059400.0  1347100.0  7852600.0   
2025-10-03  2943500.0  3619500.0  3850200.0  12948600.0  1245600.0  5755700.0   

                                                       
ticker            YUM        ZBH      ZBRA        ZTS  
date                                                   
2025-09-29  1742400.0   929000.0  482300.0  2870100.0  
2025-09-30  1898800.0  1188300.0  500700.0  3736800.0  
2025-10-01  1991500.0  1193600.0  537300.0  3677000.0  
2025-10-02  1800700.0   730500.0  450100.0  3262300.0  
2025-10-03  1288500.0   896200.0  452300.0  2569700.0  

[5 rows x 3048 columns]

In [106]:
sp500df_ohlcv_append_raw = yf.download(ticker_list,start=sp500df.index.max()+ pd.Timedelta(days=1), end=end, group_by="ticker", auto_adjust=False, threads=False)

[*********************100%***********************]  503 of 503 completed

503 Failed downloads:
['APTV', 'CF', 'MMC', 'PCAR', 'CMS', 'JCI', 'IFF', 'WY', 'VLO', 'MCD', 'WRB', 'WAB', 'CVS', 'FANG', 'TPL', 'PEG', 'DOV', 'RL', 'OMC', 'AWK', 'DXCM', 'ALLE', 'PGR', 'CI', 'CMCSA', 'LEN', 'WMB', 'VRTX', 'RSG', 'TSCO', 'NWS', 'TYL', 'CTSH', 'GLW', 'PFE', 'WEC', 'AXP', 'TRV', 'AMAT', 'HST', 'EXPD', 'META', 'PNR', 'CAG', 'COP', 'AMGN', 'EBAY', 'RTX', 'HSY', 'BRO', 'HSIC', 'CDNS', 'WDC', 'MHK', 'RJF', 'ES', 'KLAC', 'HPQ', 'ADI', 'DVA', 'MPC', 'SJM', 'UNH', 'BK', 'MMM', 'CAH', 'DGX', 'TKO', 'TJX', 'AON', 'CCI', 'CHRW', 'EMR', 'NOW', 'BLK', 'IR', 'PLD', 'NKE', 'SYK', 'WST', 'MSCI', 'COR', 'BA', 'DOW', 'ED', 'ZBH', 'CAT', 'GEN', 'VTRS', 'MDT', 'GDDY', 'CDW', 'KIM', 'REG', 'BSX', 'BKNG', 'BEN', 'KHC', 'UNP', 'NEE', 'GIS', 'CBRE', 'OKE', 'TGT', 'ALL', 'MRNA', 'BX', 'JKHY', 'ERIE', 'CARR', 'BDX', 'GEHC', 'PHM', 'IBKR', 'HII', 'CNP', 'GRMN', 'EXC', 'MO', 'INTC', 'PKG', 'NVR', 'TDG', 'FTV', 'FCX', 'AIG', 

In [11]:
sp500df_ohlcv_append_flat = (
    sp500df_ohlcv_append_raw
    .stack(level=0, future_stack=True)
    .rename_axis(["date", "ticker"])
    .reset_index()
    .rename(columns={
        "Date": "date",
        "Ticker": "ticker",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Adj Close": "adj_close",
        "Volume": "volume"
    })
    .query("date < @end")
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

sp500df_ohlcv_append_flat.columns.name = None

sp500df_ohlcv_append_flat.tail()

,date,ticker,open,high,low,close,adj_close,volume
498,2025-10-06,XYZ,77.550003,78.800003,76.690002,77.779999,77.779999,5318100
499,2025-10-06,YUM,150.089996,150.669998,148.600006,148.779999,148.779999,1395000
500,2025-10-06,ZBH,101.029999,101.309998,98.779999,98.830002,98.830002,1065500
501,2025-10-06,ZBRA,308.079987,309.079987,298.959991,301.940002,301.940002,558200
502,2025-10-06,ZTS,146.429993,147.039993,144.850006,145.360001,145.360001,3114900


In [12]:
sp500df_ohlcv_append_flat.to_sql("sp500_ohlcv", engine, if_exists="append", index=False)

503

In [74]:
sp500df = pd.read_sql("SELECT * FROM sp500_ohlcv;", engine, parse_dates=["date"]).pivot(index="date", columns="ticker", values=["open", "high", "low", "close", "adj_close", "volume"]).sort_index(axis=1, level=0)
sp500df.tail()

adj_close                                                         \
ticker             A   AAPL   ABBV   ABNB    ABT  ACGL    ACN   ADBE    ADI   
date                                                                          
2025-09-30    128.35 254.63 231.54 121.42 133.94 90.73 246.60 352.75 245.70   
2025-10-01    138.58 255.45 244.38 122.32 133.47 90.31 243.71 343.72 239.28   
2025-10-02    138.70 257.13 236.56 121.49 132.99 89.08 244.34 351.48 241.67   
2025-10-03    141.64 258.02 233.91 120.22 134.59 90.79 245.32 346.74 241.99   
2025-10-06    141.61 256.69 230.19 120.35 133.74 91.35 248.17 350.14 242.50   

                  ...       volume                                          \
ticker       ADM  ...           WY         WYNN          XEL           XOM   
date              ...                                                        
2025-09-30 59.74  ... 5,465,100.00 1,511,300.00 4,011,000.00 18,076,200.00   
2025-10-01 59.25  ... 3,650,300.00 1,532,600.00 4,585,600.00 16,614,000.00   
2025-10-02 59.11  ... 3,592,700.00 1,286,100.00 6,982,500.00 13,059,400.00   
2025-10-03 61.04  ... 2,943,500.00 3,619,500.00 3,850,200.00 12,948,600.00   
2025-10-06 62.45  ... 3,680,500.00 1,799,500.00 4,082,600.00 12,034,200.00   

                                                                           \
ticker              XYL          XYZ          YUM          ZBH       ZBRA   
date                                                                        
2025-09-30 1,841,000.00 7,838,100.00 1,898,800.00 1,188,300.00 500,700.00   
2025-10-01 1,321,000.00 6,640,700.00 1,991,500.00 1,193,600.00 537,300.00   
2025-10-02 1,347,100.00 7,852,600.00 1,800,700.00   730,500.00 450,100.00   
2025-10-03 1,245,600.00 5,755,700.00 1,288,500.00   896,200.00 452,300.00   
2025-10-06 1,330,600.00 5,318,100.00 1,395,000.00 1,065,500.00 558,200.00   

                         
ticker              ZTS  
date                     
2025-09-30 3,736,800.00  
2025-10-01 3,677,000.00  
2025-10-02 3,262,300.00  
2025-10-03 2,569,700.00  
2025-10-06 3,114,900.00  

[5 rows x 3048 columns]

In [75]:
daily_returns = sp500df["adj_close"].pct_change(fill_method=None).dropna(how="all")
daily_returns.tail()

ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
date,,,,,,,,,,,,,,,,,,,,,
2025-09-30,0.04,0.00,0.04,-0.01,0.01,0.01,-0.00,-0.02,0.00,-0.01,...,-0.00,-0.03,0.01,-0.01,0.02,-0.04,-0.01,0.00,0.00,0.02
2025-10-01,0.08,0.00,0.06,0.01,-0.00,-0.00,-0.01,-0.03,-0.03,-0.01,...,0.01,0.03,-0.00,-0.01,0.00,0.02,0.01,0.00,-0.02,0.00
2025-10-02,0.00,0.01,-0.03,-0.01,-0.00,-0.01,0.00,0.02,0.01,-0.00,...,-0.00,0.01,-0.01,-0.01,0.01,0.05,-0.01,0.00,0.01,-0.00
2025-10-03,0.02,0.00,-0.01,-0.01,0.01,0.02,0.00,-0.01,0.00,0.03,...,0.01,-0.07,0.01,0.02,0.01,0.00,-0.00,0.02,0.03,-0.00
2025-10-06,-0.00,-0.01,-0.02,0.00,-0.01,0.01,0.01,0.01,0.00,0.02,...,-0.01,0.01,0.01,0.01,0.00,0.01,-0.01,-0.02,-0.01,-0.01


In [76]:
returns_1d = daily_returns.iloc[-1].dropna().sort_values(ascending=False)
returns_1d.head(10), returns_1d.tail(10)

(ticker
 AMD    0.24
 TSLA   0.05
 MPWR   0.05
 SMCI   0.05
 ALB    0.04
 HUM    0.04
 DASH   0.04
 PLTR   0.04
 DDOG   0.04
 UBER   0.04
 Name: 2025-10-06 00:00:00, dtype: float64,
 ticker
 CLX    -0.04
 STX    -0.04
 SW     -0.04
 CSGP   -0.04
 T      -0.04
 ARE    -0.05
 WDC    -0.05
 SBUX   -0.05
 VZ     -0.05
 APP    -0.14
 Name: 2025-10-06 00:00:00, dtype: float64)

In [ ]:
lookback_map = {
    "1D": 1,
    "1W": 5,
    "1M": 21,
    "3M": 63,
    "6M": 126
}

choice = input("Enter lookback horizon (1D, 1W, 1M, 3M, 6M, ALL, or number of trading days): ").upper()

if choice in lookback_map:
    window = lookback_map[choice]
    returns = (
        sp500df["adj_close"]
        .pct_change(window, fill_method=None)
        .iloc[-1]
        .dropna()
        .sort_values(ascending=False)
    )
    display(returns.head(10), returns.tail(10))

elif choice == "ALL":
    full_cum_returns = (
        sp500df["adj_close"].iloc[-1] / sp500df["adj_close"].iloc[0] - 1
    ).dropna().sort_values(ascending=False)
    display(full_cum_returns.head(10), full_cum_returns.tail(10))

elif choice.isdigit():
    window = int(choice)
    if len(sp500df) > window:
        returns = (
            sp500df["adj_close"]
            .pct_change(window, fill_method=None)
            .iloc[-1]
            .dropna()
            .sort_values(ascending=False)
        )
        display(returns.head(10), returns.tail(10))
    else:
        print(f"Not enough rows in dataset for {window} trading days.")

else:
    print("Invalid choice. Please enter 1D, 1W, 1M, 3M, 6M, ALL, or a number of trading days.")

In [77]:
fig = go.Figure()

for ticker in daily_returns.columns:
    fig.add_trace(go.Scatter(
        x=daily_returns.index,
        y=daily_returns[ticker],
        mode='lines',
        name=ticker,
        customdata=[[ticker]] * len(daily_returns),
        hovertemplate=(
        "Date: %{x}<br>" +
        "Return: %{y:.4f}<br>" +
        "Ticker: %{customdata[0]}<extra></extra>"
        )
    ))

fig.update_layout(
    title='Daily Returns of S&P 500 Tickers',
    xaxis_title='Date',
    yaxis_title='Daily Return',
    template='plotly_dark',
    xaxis=dict(rangeslider=dict(visible=True))
)

fig.show(config={'displaylogo': False})

In [99]:
user_input_stock_symbol = input("Enter stock symbol to inspect (e.g., 'AAPL', 'MSFT'): ").upper()

open = sp500df["open"][user_input_stock_symbol]
high = sp500df["high"][user_input_stock_symbol]
low = sp500df["low"][user_input_stock_symbol]
close = sp500df["close"][user_input_stock_symbol]
volume = sp500df["volume"][user_input_stock_symbol]
adj_close = sp500df["adj_close"][user_input_stock_symbol]

ticker = pd.DataFrame({
    "date": adj_close.index,
    "open": open.values,
    "high": high.values,
    "low": low.values,
    "close": close.values,
    "adj_close": adj_close.values,
    "volume": volume.values,
})

ticker["returns"] = ticker["adj_close"].pct_change(fill_method=None)
ticker.reset_index(drop=True, inplace=True)
ticker.tail()

,date,open,high,low,close,adj_close,volume,returns
185,2025-09-30,76.46,76.70,75.12,75.49,75.49,"9,392,700.00",-0.01
186,2025-10-01,76.33,78.70,76.22,78.67,78.67,"15,044,800.00",0.04
187,2025-10-02,78.64,78.89,77.54,78.18,78.18,"9,263,900.00",-0.01
188,2025-10-03,78.45,81.36,77.65,80.06,80.06,"12,115,900.00",0.02
189,2025-10-06,80.15,82.37,80.15,82.11,82.11,"15,762,700.00",0.03


In [100]:
lookback_map = {
    "1D": 1,
    "1W": 5,
    "1M": 21,
    "3M": 63,
    "6M": 126
}

choice = input("Enter lookback horizon (1D, 1W, 1M, 3M, 6M, ALL, or number of trading days): ").upper()

if choice in lookback_map:
    window = lookback_map[choice]
    if len(ticker) > window:
        ret = ticker["adj_close"].iloc[-1] / ticker["adj_close"].iloc[-window-1] - 1
        print(f"{choice} return for {user_input_stock_symbol}: {ret:.2%}")
    else:
        print(f"Not enough data for {choice}")

elif choice == "ALL":
    ret = ticker["adj_close"].iloc[-1] / ticker["adj_close"].iloc[0] - 1
    print(f"Total return for {user_input_stock_symbol}: {ret:.2%}")

elif choice.isdigit():
    window = int(choice)
    if len(ticker) > window:
        ret = ticker["adj_close"].iloc[-1] / ticker["adj_close"].iloc[-window-1] - 1
        print(f"{window}-day return for {user_input_stock_symbol}: {ret:.2%}")
    else:
        print(f"Not enough data for {window} trading days")

else:
    print("Invalid choice. Please enter 1D, 1W, 1M, 3M, 6M, ALL, or a number of trading days.")

45-day return for NEE: 16.63%


In [101]:
fig = go.Figure()

fig.add_trace(go.Candlestick(
    x=ticker["date"], open=ticker["open"], high=ticker["high"], low=ticker["low"], close=ticker["close"], name="Candlestick"
))

fig.add_trace(go.Bar(
    x=ticker["date"], y=ticker["volume"], name="Volume", marker=dict(color="gray"), opacity=0.3, yaxis="y2"
))

fig.update_layout(
    title=f"{user_input_stock_symbol} Candlestick and Volume",
    xaxis=dict(title="Date", tickformat="%b %d"),
    yaxis=dict(title="Price ($)"),
    yaxis2=dict(title="Volume", overlaying="y", side="right", showgrid=False, color="gray"),
    legend=dict(x=0.01, y=0.99, bordercolor="black", borderwidth=1),
    bargap=0,
    template="plotly_dark",
    width=1200,
    height=500
)

fig.show(config={'displaylogo': False})


In [102]:
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f"{user_input_stock_symbol} Daily Returns 2025",
    f"{user_input_stock_symbol} Daily Returns Distribution"
))

fig.add_trace(
    go.Scatter(x=ticker["date"], y=ticker["returns"], mode="lines+markers",
               line=dict(color="blue", width=0.5),
               marker=dict(symbol="triangle-down", size=4),
               name="Daily Returns"),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(x=ticker["returns"].dropna(), nbinsx=25,
                 marker=dict(color="blue", line=dict(color="black", width=1)),
                 opacity=0.7, name="Distribution"),
    row=1, col=2
)

fig.update_layout(
    template="plotly_dark",
    width=1200, height=500,
    showlegend=True
)

fig.update_xaxes(title_text="Date", tickformat="%b %d", row=1, col=1)
fig.update_yaxes(title_text="% Change", row=1, col=1)

fig.update_xaxes(title_text="Daily Return (% Change)", row=1, col=2)
fig.update_yaxes(title_text="Frequency", row=1, col=2)

fig.show(config={'displaylogo': False})

In [82]:
fasb_fetch_year = datetime.datetime.now().year
url = f"https://xbrl.fasb.org/us-gaap/{fasb_fetch_year}/us-gaap-{fasb_fetch_year}.zip"

resp = requests.get(url)
resp.raise_for_status()

elements = []

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    xsd_files = [name for name in z.namelist() if name.startswith(f"us-gaap-{fasb_fetch_year}/elts/") and name.endswith(".xsd")]
    
    for xsd_file in xsd_files:
        with z.open(xsd_file) as f:
            try:
                tree = ET.parse(f)
                root = tree.getroot()
                for elem in root.findall(".//{http://www.w3.org/2001/XMLSchema}element"):
                    elements.append({
                        "id": elem.get("id"),
                        "name": elem.get("name"),
                        "type": elem.get("type"),
                        "substitutionGroup": elem.get("substitutionGroup"),
                        "balance": elem.get("balance"),
                        "periodType": elem.get("periodType"),
                        "source": xsd_file
                    })
            except Exception:
                # Skip non-XML or problematic files
                continue

xbrl_taxonomy = pd.DataFrame(elements).drop_duplicates(subset=["id"])

In [83]:
xbrl_taxonomy.to_sql("xbrl_taxonomy", engine, if_exists="replace", index=False)

355

In [84]:
xbrl_taxonomy = pd.read_sql("SELECT * FROM xbrl_taxonomy;", engine)
xbrl_taxonomy

,id,name,type,substitutionGroup,balance,periodType,source
0,us-gaap_AccidentAndHealthInsuranceSegmentMember,AccidentAndHealthInsuranceSegmentMember,dtr-types:domainItemType,xbrli:item,None,None,us-gaap-2025/elts/us-gaap-2025.xsd
1,us-gaap_OtherAccountsPayableAndAccruedLiabilities,OtherAccountsPayableAndAccruedLiabilities,xbrli:monetaryItemType,xbrli:item,None,None,us-gaap-2025/elts/us-gaap-2025.xsd
2,us-gaap_AccountingForCertainLoansAndDebtSecuri...,AccountingForCertainLoansAndDebtSecuritiesAcqu...,dtr-types:textBlockItemType,xbrli:item,None,None,us-gaap-2025/elts/us-gaap-2025.xsd
3,us-gaap_InterestsContinuedToBeHeldByTransferor...,InterestsContinuedToBeHeldByTransferorInFinanc...,xbrli:stringItemType,xbrli:item,None,None,us-gaap-2025/elts/us-gaap-2025.xsd
4,us-gaap_InterestsContinuedToBeHeldByTransferor...,InterestsContinuedToBeHeldByTransferorInFinanc...,dtr-types:textBlockItemType,xbrli:item,None,None,us-gaap-2025/elts/us-gaap-2025.xsd
...,...,...,...,...,...,...,...
17350,tin-part_URI,URI,xs:anyURI,link:part,None,None,us-gaap-2025/elts/us-parts-tin-2025.xsd
17351,tin-part_Source_ASU_Number,Source_ASU_Number,tin-part:AsuNumber,link:part,None,None,us-gaap-2025/elts/us-parts-tin-2025.xsd
17352,tin-part_inlineURI,inlineURI,xs:anyURI,link:part,None,None,us-gaap-2025/elts/us-parts-tin-2025.xsd
17353,tin-part_pdfURI,pdfURI,xs:anyURI,link:part,None,None,us-gaap-2025/elts/us-parts-tin-2025.xsd


In [86]:
xbrl_query_ids = [
    "us-gaap_Assets",
    "us-gaap_AssetsCurrent",
    "us-gaap_AssetsNoncurrent",

    "us-gaap_Liabilities",
    "us-gaap_LiabilitiesCurrent",
    "us-gaap_LiabilitiesNoncurrent",

    "us-gaap_StockholdersEquity",
    "us-gaap_StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
    "us-gaap_LiabilitiesAndStockholdersEquity",

    "us-gaap_RevenueFromContractWithCustomerExcludingAssessedTax",
    "us-gaap_SalesRevenueNet",
    "us-gaap_Revenues",
    "us-gaap_RevenueFromContractWithCustomerIncludingAssessedTax",

    "us-gaap_GrossProfit",
    "us-gaap_GrossProfitAbstract",

    "us-gaap_OperatingIncomeLoss",

    "us-gaap_NetIncomeLoss",
    "us-gaap_ProfitLoss",
    "us-gaap_NetIncomeLossAvailableToCommonStockholdersBasic",
    "us-gaap_NetIncomeLossAvailableToCommonStockholdersDiluted",
    "us-gaap_NetIncomeLossAttributableToNoncontrollingInterest",
    "us-gaap_NetIncomeLossAttributableToParent",
    "us-gaap_NetIncomeLossAttributableToRedeemableNoncontrollingInterest",
    "us-gaap_NetIncomeLossAttributableToNonredeemableNoncontrollingInterest",
    "us-gaap_NetIncomeLossIncludingPortionAttributableToNoncontrollingInterest",
    "us-gaap_NetIncomeLossAbstract",

    "us-gaap_EarningsPerShareBasic",
    "us-gaap_EarningsPerShareDiluted",

    "us-gaap_NetCashProvidedByUsedInOperatingActivities",
    "us-gaap_NetCashProvidedByUsedInInvestingActivities",
    "us-gaap_NetCashProvidedByUsedInFinancingActivities",
    "us-gaap_PaymentsToAcquirePropertyPlantAndEquipment",

    "us-gaap_WeightedAverageNumberOfSharesOutstandingBasic",
    "us-gaap_WeightedAverageNumberOfDilutedSharesOutstanding"
]

In [88]:
with engine.connect() as conn:
    last_filed = pd.read_sql(
        "SELECT MAX(filed) as max_date FROM sp500_edgar_financials", conn
    ).iloc[0]["max_date"]

last_filed = pd.to_datetime(last_filed, errors="coerce").normalize()

id_to_name = dict(zip(xbrl_taxonomy["id"], xbrl_taxonomy["name"]))

sp500_edgar_financials_append = []

pbar = tqdm(sp500_reference.iterrows(), total=len(sp500_reference), desc="Updating financial metrics")

for _, row in pbar:
    ticker = row["ticker"]
    cik = str(row["cik"]).zfill(10)
    
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    r = requests.get(url, headers={"User-Agent": "dbater1993@gmail.com"})
    if r.status_code != 200:
        continue
    
    financial_metrics = r.json().get("facts", {}).get("us-gaap", {})
    
    for xid in xbrl_query_ids:
        xname = id_to_name.get(xid, xid.replace("us-gaap_", ""))
        
        metric_data = financial_metrics.get(xname)
        if not metric_data:
            continue
        
        for unit, datapoints in metric_data.get("units", {}).items():
            for dp in datapoints:
                filed_date = pd.to_datetime(dp.get("filed"), errors="coerce").normalize()
                end_date = pd.to_datetime(dp.get("end"), errors="coerce").normalize()
                
                if pd.isna(filed_date):
                    continue  # skip if no valid filed date
                
                if pd.isna(last_filed) or filed_date > last_filed:
                    sp500_edgar_financials_append.append({
                        "ticker": ticker,
                        "cik": cik,
                        "id": xid,
                        "metric": xname,
                        "unit": unit,
                        "value": dp.get("val"),
                        "fy": dp.get("fy"),
                        "fp": dp.get("fp"),
                        "form": dp.get("form"),
                        "filed": filed_date,
                        "end": end_date
                    })
    
    pbar.set_description(f"Collected: {len(sp500_edgar_financials_append)} new rows")
    time.sleep(0.2)

pbar.close()

sp500_edgar_financials_append

Collected: 224 new rows: 100%|██████████| 503/503 [31:59<00:00,  3.82s/it]


[{'ticker': 'STZ',
  'cik': '0000016918',
  'id': 'us-gaap_Assets',
  'metric': 'Assets',
  'unit': 'USD',
  'value': 21652300000,
  'fy': 2026,
  'fp': 'Q2',
  'form': '10-Q',
  'filed': Timestamp('2025-10-07 00:00:00'),
  'end': Timestamp('2025-02-28 00:00:00')},
 {'ticker': 'STZ',
  'cik': '0000016918',
  'id': 'us-gaap_Assets',
  'metric': 'Assets',
  'unit': 'USD',
  'value': 21419400000,
  'fy': 2026,
  'fp': 'Q2',
  'form': '10-Q',
  'filed': Timestamp('2025-10-07 00:00:00'),
  'end': Timestamp('2025-08-31 00:00:00')},
 {'ticker': 'STZ',
  'cik': '0000016918',
  'id': 'us-gaap_AssetsCurrent',
  'metric': 'AssetsCurrent',
  'unit': 'USD',
  'value': 3716400000,
  'fy': 2026,
  'fp': 'Q2',
  'form': '10-Q',
  'filed': Timestamp('2025-10-07 00:00:00'),
  'end': Timestamp('2025-02-28 00:00:00')},
 {'ticker': 'STZ',
  'cik': '0000016918',
  'id': 'us-gaap_AssetsCurrent',
  'metric': 'AssetsCurrent',
  'unit': 'USD',
  'value': 2878900000,
  'fy': 2026,
  'fp': 'Q2',
  'form': '10-Q',

In [89]:
sp500_edgar_financials_append = pd.DataFrame(sp500_edgar_financials_append)
sp500_edgar_financials_append.to_sql("sp500_edgar_financials", engine, if_exists="append", index=False)

224

In [103]:
user_stock_fundamentals_query = f"""
SELECT "end", id, metric, value
FROM sp500_edgar_financials
WHERE ticker = '{user_input_stock_symbol}' AND form = '10-K'
ORDER BY "end";
"""
user_stock_fundamentals_query_df = pd.read_sql(user_stock_fundamentals_query, engine)
user_stock_fundamentals_query_df

,end,id,metric,value
0,2006-12-31,us-gaap_StockholdersEquity,StockholdersEquity,"9,930,000,000.00"
1,2007-12-31,us-gaap_StockholdersEquity,StockholdersEquity,"10,735,000,000.00"
2,2007-12-31,us-gaap_StockholdersEquity,StockholdersEquity,"10,735,000,000.00"
3,2007-12-31,us-gaap_WeightedAverageNumberOfSharesOutstandi...,WeightedAverageNumberOfSharesOutstandingBasic,"397,700,000.00"
4,2007-12-31,us-gaap_WeightedAverageNumberOfDilutedSharesOu...,WeightedAverageNumberOfDilutedSharesOutstanding,"400,600,000.00"
...,...,...,...,...
1219,2024-12-31,us-gaap_EarningsPerShareDiluted,EarningsPerShareDiluted,3.37
1220,2024-12-31,us-gaap_Liabilities,Liabilities,"129,283,000,000.00"
1221,2024-12-31,us-gaap_ProfitLoss,ProfitLoss,"5,698,000,000.00"
1222,2024-12-31,us-gaap_NetIncomeLossAttributableToNonredeemab...,NetIncomeLossAttributableToNonredeemableNoncon...,"-1,248,000,000.00"


In [104]:
user_stock_fundamentals_query_df["end"] = pd.to_datetime(user_stock_fundamentals_query_df["end"])

pivot = user_stock_fundamentals_query_df.pivot_table(
    index="metric",
    columns=user_stock_fundamentals_query_df["end"].dt.year,
    values="value",
    aggfunc="last"
)

growth = (pivot - pivot.shift(axis=1)) / pivot.shift(axis=1).abs()

growth_clipped = growth.clip(lower=-1, upper=1) * 100   # -100% to +100%

metrics = pivot.index.tolist()
metrics_sorted = sorted([m for m in metrics if m != "Assets"])
if "Assets" in metrics:
    metrics_sorted = ["Assets"] + metrics_sorted

pivot = pivot.loc[metrics_sorted]
growth_clipped = growth_clipped.loc[metrics_sorted]

colorscale = [
    [0.00, "#67001f"], [0.05, "#b2182b"], [0.10, "#d6604d"],
    [0.20, "#f4a582"], [0.30, "#fddbc7"], [0.40, "#f7f7f7"],
    [0.50, "#d1e5f0"], [0.60, "#92c5de"], [0.70, "#4393c3"],
    [0.80, "#2166ac"], [1.00, "#053061"]
]

fig = go.Figure(
    data=go.Heatmap(
        z=growth_clipped.fillna(0).values,
        x=growth_clipped.columns,
        y=growth_clipped.index,
        colorscale=colorscale,
        zmin=-100,
        zmax=100,
        colorbar=dict(title="YoY % Growth"),
        customdata=pivot.fillna("").values,
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Year: %{x}<br>"
            "Value: %{customdata:,}<br>"
            "Growth: %{z:.2f}%<extra></extra>"
        )
    )
)

fig.update_layout(
    title=f"{user_input_stock_symbol} Fundamentals Growth Heatmap (10-K)",
    xaxis_title="Year",
    yaxis_title="Metric",
    height=800,
    yaxis=dict(categoryorder="array", categoryarray=metrics_sorted),
    template="plotly_dark"
)

fig.update_yaxes(autorange="reversed")
fig.show(config={'displaylogo': False})

In [92]:
pd.options.display.float_format = '{:,.2f}'.format

sectors_query = "SELECT DISTINCT sector FROM sp500_reference ORDER BY sector;"

latest_10k_query = """
WITH latest_end_per_ticker AS (
    SELECT
        ticker,
        MAX("end") AS latest_end
    FROM sp500_edgar_financials
    GROUP BY ticker
)
SELECT 
    f.ticker,
    f.metric,
    f.value,
    f.end
FROM sp500_edgar_financials f
JOIN latest_end_per_ticker l
    ON f.ticker = l.ticker
   AND f.end = l.latest_end
ORDER BY f.ticker, f.metric;
"""

latest_10k_df = pd.read_sql(latest_10k_query, engine)
latest_10k_df['end'] = pd.to_datetime(latest_10k_df['end'], errors='coerce')

latest_10k_df = latest_10k_df.merge(
    sp500_reference[['ticker', 'sector']],
    on='ticker',
    how='left'
)

latest_10k_df

,ticker,metric,value,end,sector
0,A,Assets,"12,226,000,000.00",2025-07-31,Health Care
1,A,AssetsCurrent,"4,253,000,000.00",2025-07-31,Health Care
2,A,EarningsPerShareBasic,1.18,2025-07-31,Health Care
3,A,EarningsPerShareBasic,3.05,2025-07-31,Health Care
4,A,EarningsPerShareDiluted,3.05,2025-07-31,Health Care
...,...,...,...,...,...
13424,ZTS,StockholdersEquityIncludingPortionAttributable...,"4,977,000,000.00",2025-06-30,Health Care
13425,ZTS,WeightedAverageNumberOfDilutedSharesOutstanding,"445,500,000.00",2025-06-30,Health Care
13426,ZTS,WeightedAverageNumberOfDilutedSharesOutstanding,"446,700,000.00",2025-06-30,Health Care
13427,ZTS,WeightedAverageNumberOfSharesOutstandingBasic,"445,100,000.00",2025-06-30,Health Care


In [93]:
sp500df_flat = (
    sp500df['adj_close']
    .stack()
    .reset_index()
    .rename(columns={
        'date': 'date',
        'ticker': 'ticker',
        0: 'adj_close'
    })
    .sort_values(['ticker', 'date'])
    .groupby('ticker', group_keys=False)
    .tail(2)
    .reset_index(drop=True)
)

shares_out = latest_10k_df[
    latest_10k_df['metric'].str.strip().str.lower() ==
    "weightedaveragenumberofdilutedsharesoutstanding"
][['ticker', 'value', 'sector']].copy()

merged = sp500df_flat.merge(shares_out, on='ticker', how='inner')

merged['market_cap'] = merged['adj_close'] * merged['value']

merged = (
    merged.sort_values(['ticker', 'date'])
    .assign(pct_change=lambda df: df.groupby('ticker')['market_cap'].pct_change())
)

marketcap_latest = (
    merged.groupby('ticker', as_index=False)
    .last()
    .dropna(subset=['pct_change'])
)

marketcap_latest

,ticker,date,adj_close,value,sector,market_cap,pct_change
0,A,2025-10-06,141.61,"285,000,000.00",Health Care,"40,358,850,173.95",0.00
1,AAPL,2025-10-06,256.69,"15,051,726,000.00",Information Technology,"3,863,627,583,687.38",0.01
2,ABBV,2025-10-06,230.19,"1,771,000,000.00",Health Care,"407,666,494,323.73",-0.00
3,ABNB,2025-10-06,120.35,"629,000,000.00",Consumer Discretionary,"75,700,149,040.22",0.00
4,ABT,2025-10-06,133.74,"1,749,054,000.00",Health Care,"233,918,491,567.84",-0.00
...,...,...,...,...,...,...,...
484,XYZ,2025-10-06,77.78,"627,103,000.00",Financials,"48,776,070,574.49",0.01
485,YUM,2025-10-06,148.78,"282,000,000.00",Consumer Discretionary,"41,955,959,655.76",0.00
486,ZBH,2025-10-06,98.83,"198,300,000.00",Health Care,"19,597,989,363.10",-0.00
487,ZBRA,2025-10-06,301.94,"51,282,273.00",Information Technology,"15,484,169,634.82",-0.01


In [94]:
root_name = "S&P 500"

root_df = pd.DataFrame({
    "labels": [root_name],
    "parents": [""],
    "values": [marketcap_latest["market_cap"].sum()],
    "pct_change": [0],
    "adj_close": [np.nan],
    "shares": [np.nan],
})

sector_df = (
    marketcap_latest.groupby("sector", as_index=False)
    .agg({"market_cap": "sum"})
    .assign(parents=root_name, pct_change=0, adj_close=np.nan, shares=np.nan)
    .rename(columns={"sector": "labels", "market_cap": "values"})
)

company_df = marketcap_latest.rename(
    columns={
        "ticker": "labels",
        "sector": "parents",
        "market_cap": "values",
        "adj_close": "adj_close",
        "value": "shares"
    }
)[["labels", "parents", "values", "pct_change", "adj_close", "shares"]]

tree_data = pd.concat([root_df, sector_df, company_df], ignore_index=True)

fig = go.Figure(
    go.Treemap(
        labels=tree_data["labels"],
        parents=tree_data["parents"],
        values=tree_data["values"],
        customdata=tree_data[["adj_close", "shares"]],
        marker=dict(
            colors=tree_data["pct_change"],
            colorscale=[(0, "red"), (0.5, "lightgray"), (1, "green")],
            cmin=-0.05,
            cmax=0.05,
            showscale=True,
            colorbar=dict(title="% Change", tickformat=".2%")
        ),
        hovertemplate=(
            "<b>%{label}</b><br>"
            "Parent: %{parent}<br>"
            "Market Cap: $%{value:,.0f}<br>"
            "Adj Close: $%{customdata[0]:,.2f}<br>"
            "Shares: %{customdata[1]:,.0f}<br>"
            "% Change: %{color:.2%}<extra></extra>"
        ),
        branchvalues="total",
    )
)

latest_date = marketcap_latest["date"].max().strftime("%B %d, %Y")
fig.update_layout(
    title=f"S&P 500 Market Cap Heatmap – {latest_date}",
    template="plotly_dark",
    height=900,
    margin=dict(t=80, l=40, r=40, b=40),
)

fig.show(config={"displaylogo": False})

In [20]:
revenue_summary = (
    latest_10k_df[latest_10k_df['metric'].str.lower() == "earningspersharediluted"]
    .groupby('sector')['value']
    .agg(['count', 'sum', 'mean', 'median'])
    .sort_values('mean', ascending=False)
    .reset_index()
)
revenue_summary

,sector,count,sum,mean,median
0,Consumer Discretionary,93,719.22,7.73,1.91
1,Financials,141,549.27,3.90,2.81
2,Industrials,147,545.69,3.71,2.42
3,Health Care,114,416.79,3.66,2.29
4,Communication Services,44,141.35,3.21,1.97
5,Information Technology,122,373.52,3.06,1.99
6,Energy,42,100.91,2.40,1.92
7,Materials,51,101.69,1.99,2.01
8,Consumer Staples,58,91.27,1.57,1.33
9,Utilities,58,87.09,1.50,1.19


In [21]:
pd.options.display.float_format = '{:,.2f}'.format

sector_name = input("Enter sector name (e.g., 'Information Technology'): ").title()

sector_metric_summary = (
    latest_10k_df[latest_10k_df['sector'] == sector_name]
    .groupby('metric')['value']
    .agg(['count', 'sum', 'mean', 'median'])
    .sort_values('mean', ascending=False)
    .reset_index()
)

sector_metric_summary

,metric,count,sum,mean,median
0,AssetsNoncurrent,6,"460,322,055,000.00","76,720,342,500.00","38,606,777,500.00"
1,Assets,68,"3,306,728,421,000.00","48,628,359,132.35","16,473,100,000.00"
2,LiabilitiesAndStockholdersEquity,68,"3,306,728,421,000.00","48,628,359,132.35","16,473,100,000.00"
3,LiabilitiesNoncurrent,9,"288,743,434,000.00","32,082,603,777.78","10,887,000,000.00"
4,Liabilities,54,"1,426,761,828,000.00","26,421,515,333.33","8,746,500,000.00"
5,StockholdersEquity,64,"1,296,924,183,000.00","20,264,440,359.38","6,579,396,500.00"
6,StockholdersEquityIncludingPortionAttributable...,24,"431,724,525,000.00","17,988,521,875.00","8,948,600,000.00"
7,Revenues,31,"555,182,570,000.00","17,909,115,161.29","10,360,000,000.00"
8,AssetsCurrent,68,"1,122,912,110,000.00","16,513,413,382.35","7,563,900,000.00"
9,RevenueFromContractWithCustomerExcludingAssess...,96,"1,326,491,410,000.00","13,817,618,854.17","3,192,350,000.00"


In [79]:
sector_breakdown = (
    sp500_reference
    .groupby('sector')['ticker']
    .count()
    .sort_values(ascending=False)
    .reset_index(name='count')
)

sector_breakdown

,sector,count
0,Industrials,79
1,Financials,75
2,Information Technology,68
3,Health Care,60
4,Consumer Discretionary,50
5,Consumer Staples,37
6,Utilities,31
7,Real Estate,31
8,Materials,26
9,Communication Services,24


In [36]:
fama_french_url = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip'

response = requests.get(fama_french_url)

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    file_name = z.namelist()[0]
    with z.open(file_name) as file:
        fama_french_five_factor = pd.read_csv(file,index_col=0, parse_dates=True,skiprows=3)

fama_french_five_factor = fama_french_five_factor.iloc[:-1]

fama_french_five_factor.reset_index(inplace=True)

fama_french_five_factor.columns = ['Date', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']

fama_french_five_factor['Date'] = pd.to_datetime(fama_french_five_factor['Date'])

fama_french_five_factor = fama_french_five_factor[
    (fama_french_five_factor['Date'] >= pd.to_datetime(ohlcv_start).normalize()) &
    (fama_french_five_factor['Date'] <= pd.to_datetime(ohlcv_end).normalize())
]

fama_french_five_factor.reset_index(drop=True, inplace=True)

fama_french_five_factor

,Date,Mkt-RF,SMB,HML,RMW,CMA,RF
0,2025-01-02,-0.16,-0.02,-0.32,-0.65,0.61,0.02
1,2025-01-03,1.34,0.29,-0.94,-1.26,0.37,0.02
2,2025-01-06,0.55,-0.62,-0.33,-0.40,0.55,0.02
3,2025-01-07,-1.18,0.13,0.84,0.20,-1.05,0.02
4,2025-01-08,0.10,-0.58,-0.12,0.78,-0.02,0.02
...,...,...,...,...,...,...,...
139,2025-07-25,0.40,-0.14,-0.06,-0.08,0.14,0.02
140,2025-07-28,0.04,-0.03,-0.39,0.29,-0.57,0.02
141,2025-07-29,-0.37,-0.92,0.07,0.22,-0.04,0.02
142,2025-07-30,-0.11,-0.73,-0.84,-0.40,-0.88,0.02


In [40]:
daily_returns.index = pd.to_datetime(daily_returns.index)

train_start = fama_french_five_factor['Date'].min()
train_end = fama_french_five_factor['Date'].max()

excess_returns = daily_returns.loc[
    daily_returns.index.intersection(fama_french_five_factor['Date'])
].sub(
    fama_french_five_factor.set_index("Date").loc[daily_returns.index.intersection(fama_french_five_factor['Date']), "RF"] / 100,
    axis=0
)

train_returns = excess_returns.loc[train_start:train_end]
train_factors = fama_french_five_factor.set_index("Date").loc[train_start:train_end, ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']]

regression_results = []

for ticker in train_returns.columns:
    y_train = train_returns[ticker]
    X_train = sm.add_constant(train_factors)

    common_idx = y_train.index.intersection(X_train.index)
    y_train = y_train.loc[common_idx]
    X_train = X_train.loc[common_idx]

    model = sm.OLS(y_train, X_train).fit()

    result = {
        'Ticker': ticker,
        'α': model.params['const'],
        'Mkt-RF': model.params.get('Mkt-RF', None),
        'SMB': model.params.get('SMB', None),
        'HML': model.params.get('HML', None),
        'RMW': model.params.get('RMW', None),
        'CMA': model.params.get('CMA', None),
        'R-squared': model.rsquared
    }
    regression_results.append(result)

regression_summary_df = pd.DataFrame(regression_results)

regression_summary_df.sort_values(by='α', ascending=False).reset_index(drop=True).head(25)


,Ticker,α,Mkt-RF,SMB,HML,RMW,CMA,R-squared
0,SMCI,0.006139,0.008137,0.002025,-0.027387,-0.021137,0.032771,0.446929
1,GEV,0.004635,0.012334,-0.000380,-0.013570,-0.007580,0.009365,0.498134
2,PLTR,0.004447,0.013965,-0.001196,-0.007490,-0.033543,-0.009241,0.557139
3,NRG,0.004296,0.015094,0.003533,-0.009990,-0.001965,0.004928,0.491137
4,STX,0.003978,0.014265,0.000771,0.002035,0.001408,0.004436,0.464426
5,DLTR,0.003694,0.008061,0.012266,0.005256,0.008222,-0.000105,0.199229
6,TPR,0.003463,0.013241,0.006106,0.001278,-0.000579,-0.002160,0.512437
7,VST,0.003324,0.014125,0.005915,-0.020059,-0.002553,0.016018,0.493555
8,DG,0.003199,-0.001488,0.004549,-0.000111,0.006584,0.007744,0.061016
9,WDC,0.003168,0.015634,-0.000075,-0.000776,-0.003683,0.002343,0.549936


In [ ]:
def regression_sort():
    sort_by = input(f"Choose a column to sort by {list(regression_summary_df.columns[1:])}: ")
    order = input("Sort ascending? (yes/no): ").lower() == "yes"
    return regression_summary_df.sort_values(by=sort_by, ascending=order).reset_index(drop=True).head(10)

In [ ]:
y_actual = train_returns[user_input_stock_symbol]
X = sm.add_constant(train_factors)
common_idx = y_actual.index.intersection(X.index)
y_actual = y_actual.loc[common_idx]
X = X.loc[common_idx]
model = sm.OLS(y_actual, X).fit()
y_pred = model.predict(X)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=y_actual.index,
    y=y_actual,
    mode='lines',
    name='Actual Excess Return',
    line=dict(width=1.5),
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Return: %{y:.4f}<extra></extra>'
))

fig.add_trace(go.Scatter(
    x=y_actual.index,
    y=y_pred,
    mode='lines',
    name='Predicted Excess Return (Line of Best Fit)',
    line=dict(width=2, dash='dash'),
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Predicted: %{y:.4f}<extra></extra>'
))

fig.update_layout(
    title=f"{user_input_stock_symbol}: Actual vs Predicted Excess Returns",
    xaxis_title="Date",
    yaxis_title="Excess Return",
    template="plotly_dark",
    height=600,
    width=1200,
    legend=dict(
        x=0.01, y=0.99,
        bgcolor='rgba(0,0,0,0)',
        bordercolor='rgba(255,255,255,0.1)'
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor='rgba(128,128,128,0.2)',
        rangeslider=dict(visible=True),
        rangeselector=dict(
            buttons=list([
                dict(count=3, label="3M", step="month", stepmode="backward"),
                dict(count=6, label="6M", step="month", stepmode="backward"),
                dict(count=12, label="1Y", step="month", stepmode="backward"),
                dict(step="all", label="All")
            ])
        )
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='rgba(128,128,128,0.2)'
    )
)

fig.show(config={'displaylogo': False})